In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import time, glob, re
from tqdm.auto import tqdm
from transformers import (
    AutoModelForCausalLM, AutoTokenizer,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling
)
from datasets import Dataset
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import resample
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)
from peft import LoraConfig, get_peft_model, PeftModel

In [ ]:
# Настройка промпта Bank, Blood, California, Car, Credit_G, Diabetes, Heart, Income, Jungle
base_output_dir = "/content/drive/MyDrive/finetuned_qwen_multitask"

DATASET_CONFIGS = {

    "Bank": {
        "source": "openml", "openml_id": 1461,
        "task_type": "binary",
        "feature_mappings": {
            'V1':'Age','V2':'Job','V3':'Martial','V4':'Education',
            'V5':'Default','V6':'Balance','V7':'Housing','V8':'Loan',
            'V9':'Contact','V10':'Day of Week','V11':'Month','V12':'Duration',
            'V13':'Campaign','V14':'Pdays','V15':'Previous','V16':'Poutcome',
            'Class':'y'
        },
        "prompt_config": {
            "task": "Predict whether a bank client will subscribe",
            "labels": ["no", "yes"],
            "entity": "subscription",
            "question": "Will this client subscribe?"
        },
    },

    "Blood": {
        "source": "openml", "openml_id": 1464,
        "task_type": "binary",
        "feature_mappings": {
            'V1':'Recency','V2':'Frequency','V3':'Monetary','V4':'Time',
            'Class':'Donated blood'
        },
        "prompt_config": {
            "task": "Predict whether a person donated blood",
            "labels": ["no", "yes"],
            "entity": "Donor",
            "question": "Did this person donate blood?"
        },
    },

    "California": {
        "source": "openml", "openml_id": 44090,
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether house price is above median",
            "labels": ["no", "yes"],
            "entity": "House",
            "question": "Is this house price above median?"
        },
    },

    "Car": {
        "source": "openml", "openml_id": 40975,
        "task_type": "multiclass",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict car evaluation",
            "labels": ["unacceptable", "acceptable", "good", "very good"],
            "entity": "Car",
            "question": "What is the evaluation of this car?"
        },
    },

    "Credit_G": {
        "source": "openml", "openml_id": 31,
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Classify credit risk as good or bad",
            "labels": ["bad", "good"],
            "entity": "Client",
            "question": "Is this client a good credit risk?"
        },
    },

    "Diabetes": {
        "source": "kaggle",
        "kaggle_dataset": "uciml/pima-indians-diabetes-database",
        "kaggle_file": "diabetes.csv",
        "target_col": "Outcome",
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether a patient has diabetes",
            "labels": ["no", "yes"],
            "entity": "Patient",
            "question": "Does this patient have diabetes?"
        },
    },

    "Heart": {
        "source": "kaggle",
        "kaggle_dataset": "fedesoriano/heart-failure-prediction",
        "kaggle_file": "heart.csv",
        "target_col": "HeartDisease",
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether a patient has heart disease",
            "labels": ["no", "yes"],
            "entity": "Patient",
            "question": "Does this patient have heart disease?"
        },
    },

    "Income": {
        "source": "openml", "openml_id": 1590,
        "task_type": "binary",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict whether a person earns more than 50K a year",
            "labels": ["<=50K", ">50K"],
            "entity": "Person",
            "question": "Does this person earn more than 50K a year?"
        },
    },

    "Jungle": {
        "source": "openml", "openml_id": 41027,
        "task_type": "multiclass",
        "feature_mappings": None,
        "prompt_config": {
            "task": "Predict the endgame result of Jungle Chess",
            "labels": ["white_win", "draw", "black_win"],
            "entity": "Game Position",
            "question": "What is the game result?"
        },
    },
}

# 1. Загрузка данных

In [ ]:
def load_openml(cfg):
    dataset = fetch_openml(data_id=cfg["openml_id"], as_frame=True, parser='auto')
    X = dataset.data
    y = dataset.target

    if cfg["feature_mappings"]:
        X = X.rename(columns=cfg["feature_mappings"])
        feature_names = list(cfg["feature_mappings"].values())[:-1]
        target_name = list(cfg["feature_mappings"].values())[-1]
    else:
        feature_names = X.columns.tolist()
        target_name = y.name

    X[target_name] = y
    if y.dtype == 'object' or y.dtype.name == 'category':
        le = LabelEncoder()
        X[target_name] = le.fit_transform(X[target_name])

    return X, feature_names, target_name

def load_kaggle(cfg):
    import kagglehub
    path = kagglehub.dataset_download(cfg["kaggle_dataset"])
    X = pd.read_csv(f"{path}/{cfg['kaggle_file']}")
    target_name   = cfg["target_col"]
    feature_names = [c for c in X.columns if c != target_name]
    le = LabelEncoder()
    X[target_name] = le.fit_transform(X[target_name])
    return X, feature_names, target_name

def split_data(df, target_name, test_size=0.2, val_size=0.25, seed=42):
    train_val_df, test_df = train_test_split(
        df,
        test_size=test_size,
        random_state=seed,
        stratify=df[target_name]
    )

    train_df, val_df = train_test_split(
        train_val_df,
        test_size=val_size,
        random_state=seed,
        stratify=train_val_df[target_name]
    )

    return train_df, val_df, test_df


def load_all_datasets():
    splits = {}
    for name, cfg in DATASET_CONFIGS.items():

        if cfg["source"] == "openml":
            df, feature_names, target_name = load_openml(cfg)
        else:
            df, feature_names, target_name = load_kaggle(cfg)
        train_df, val_df, test_df = split_data(df, target_name)
        splits[name] = (train_df, val_df, test_df, feature_names, target_name)
        print(f"Train={len(train_df)} Val={len(val_df)} Test={len(test_df)} Features={len(feature_names)}")
    return splits

In [ ]:
# Запуск загрузки всех данных
all_splits = load_all_datasets()

# Создание сводной таблицы для проверки
summary_data = []

for name, (train_df, val_df, test_df, feature_names, target_name) in all_splits.items():
    summary_data.append({
        "Dataset": name,
        "Train_Size": len(train_df),
        "Val_Size": len(val_df),
        "Test_Size": len(test_df),
        "Num_Features": len(feature_names),
        "Target_Column": target_name,
        "Target_Classes": train_df[target_name].nunique()
    })

df_summary = pd.DataFrame(summary_data)
print("Информация по всем датасетам")
display(df_summary)

Dataset: Bank
Train=27126 Val=9042 Test=9043 Features=16
Dataset: Blood
Train=448 Val=150 Test=150 Features=4
Dataset: California
Train=12380 Val=4127 Test=4127 Features=8
Dataset: Car
Train=1036 Val=346 Test=346 Features=6
Dataset: Credit_G
Train=600 Val=200 Test=200 Features=20
Dataset: Diabetes
Using Colab cache for faster access to the 'pima-indians-diabetes-database' dataset.
Train=460 Val=154 Test=154 Features=8
Dataset: Heart
Using Colab cache for faster access to the 'heart-failure-prediction' dataset.
Train=550 Val=184 Test=184 Features=11
Dataset: Income
Train=29304 Val=9769 Test=9769 Features=14
Dataset: Jungle
Train=26891 Val=8964 Test=8964 Features=6
Информация по всем датасетам


,Dataset,Train_Size,Val_Size,Test_Size,Num_Features,Target_Column,Target_Classes
0,Bank,27126,9042,9043,16,y,2
1,Blood,448,150,150,4,Donated blood,2
2,California,12380,4127,4127,8,price,2
3,Car,1036,346,346,6,class,4
4,Credit_G,600,200,200,20,class,2
5,Diabetes,460,154,154,8,Outcome,2
6,Heart,550,184,184,11,HeartDisease,2
7,Income,29304,9769,9769,14,class,2
8,Jungle,26891,8964,8964,6,class,3


# 2. Вспомогательные функции

In [ ]:
def row_to_text_template(row, feature_names):
    template_parts = []
    for feature in feature_names:
        value = row[feature]

        if isinstance(value, (int, np.integer)):
            phrase = f"The value of {feature} is {value}."
        elif isinstance(value, (float, np.floating)):
            phrase = f"The value of {feature} is {value:.2f}."
        else:
            phrase = f"The category of {feature} is {value}."

        template_parts.append(phrase)

    return " ".join(template_parts)

In [ ]:
def check_random_sample(dataset_name):

    train_df, _, _, feature_names, target_name = all_splits[dataset_name]

    print(f"Проверка датасета: {dataset_name}")

    display(train_df.head(2))

    print("\n Признаки: ")
    print(feature_names)

    sample_row = train_df.iloc[0]
    sample_text = row_to_text_template(sample_row, feature_names)
    print(f"\n {sample_text}")

check_random_sample("Bank")

Проверка датасета: Bank


,Age,Job,Martial,Education,Default,Balance,Housing,Loan,Contact,Day of Week,Month,Duration,Campaign,Pdays,Previous,Poutcome,y
10333,43,services,married,secondary,no,2478,yes,no,unknown,12,jun,157,1,-1,0,unknown,0
9723,40,blue-collar,married,primary,no,676,no,no,unknown,9,jun,74,1,-1,0,unknown,0



 Признаки: 
['Age', 'Job', 'Martial', 'Education', 'Default', 'Balance', 'Housing', 'Loan', 'Contact', 'Day of Week', 'Month', 'Duration', 'Campaign', 'Pdays', 'Previous', 'Poutcome']

 The value of Age is 43. The category of Job is services. The category of Martial is married. The category of Education is secondary. The category of Default is no. The value of Balance is 2478. The category of Housing is yes. The category of Loan is no. The category of Contact is unknown. The value of Day of Week is 12. The category of Month is jun. The value of Duration is 157. The value of Campaign is 1. The value of Pdays is -1. The value of Previous is 0. The category of Poutcome is unknown.


In [ ]:
def parse_prediction(response, labels):
    response = response.lower().strip().strip("'\"")
    response = response.rstrip('.,!? ').strip("'\"").strip()

    for i, class_name in enumerate(labels):
        class_lower = class_name.lower().strip().strip("'\"")

        if response == class_lower:
            return i

        if response.startswith(class_lower):
            return i

        if class_lower in response.split():
            return i

    print(f"Warning: Could not parse '{response}' (expected one of {labels})")
    return 0


# Тестирование по всем сконфигурированным датасетам
for ds_name, config in DATASET_CONFIGS.items():
    labels = config["prompt_config"]["labels"]

    # Точное совпадение первого класса
    response_1 = labels[0]
    pred_1 = parse_prediction(response_1, labels)
    print(f"Response: '{response_1}' -> Prediction Index: {pred_1}")

    # Совпадение последнего класса
    response_2 = labels[-1]
    pred_2 = parse_prediction(response_2, labels)
    print(f"Response: '{response_2}' -> Prediction Index: {pred_2}")

    # Ответ в виде предложения
    response_3 = f"The answer is {labels[0]}."
    pred_3 = parse_prediction(response_3, labels)
    print(f"Response: '{response_3}' -> Prediction Index: {pred_3}")

    # Некорректный ответ
    response_fail = "I don't know"
    pred_fail = parse_prediction(response_fail, labels)
    print("\n")


Response: 'no' -> Prediction Index: 0
Response: 'yes' -> Prediction Index: 1
Response: 'The answer is no.' -> Prediction Index: 0


Response: 'no' -> Prediction Index: 0
Response: 'yes' -> Prediction Index: 1
Response: 'The answer is no.' -> Prediction Index: 0


Response: 'no' -> Prediction Index: 0
Response: 'yes' -> Prediction Index: 1
Response: 'The answer is no.' -> Prediction Index: 0


Response: 'unacceptable' -> Prediction Index: 0
Response: 'very good' -> Prediction Index: 2
Response: 'The answer is unacceptable.' -> Prediction Index: 0


Response: 'bad' -> Prediction Index: 0
Response: 'good' -> Prediction Index: 1
Response: 'The answer is bad.' -> Prediction Index: 0


Response: 'no' -> Prediction Index: 0
Response: 'yes' -> Prediction Index: 1
Response: 'The answer is no.' -> Prediction Index: 0


Response: 'no' -> Prediction Index: 0
Response: 'yes' -> Prediction Index: 1
Response: 'The answer is no.' -> Prediction Index: 0


Response: '<=50K' -> Prediction Index: 0
Respon

In [ ]:
def compute_metrics(y_true, y_pred, y_prob, task_type):
    avg = "binary" if task_type == "binary" else "macro"

    metrics = {
        "F1": f1_score(y_true, y_pred, average=avg, zero_division=0),
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, average=avg, zero_division=0),
        "Recall": recall_score(y_true, y_pred, average=avg, zero_division=0),
    }

    try:
        mc  = "raise" if task_type == "binary" else "ovr"
        ypr = np.array(y_prob)
        metrics["ROC-AUC"] = roc_auc_score(
            y_true, ypr, multi_class=mc, average="macro")
    except:
        metrics["ROC-AUC"] = 0.0

    return metrics

In [ ]:
def bootstrap_metrics(y_true, y_pred, y_prob, task_type, n_iter=1000):
    scores = []
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    y_prob = np.array(y_prob)

    for i in range(n_iter):
        yt, yp, ypr = resample(y_true, y_pred, y_prob, random_state=i+1)
        try:
            m = compute_metrics(yt, yp, ypr, task_type)
            scores.append([m["ROC-AUC"], m["F1"], m["Accuracy"], m["Precision"], m["Recall"]])
        except Exception:
            continue

    if not scores:
        return {k: "0.0000±0.0000" for k in ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]}

    scores = np.array(scores)
    means, stds = scores.mean(0), scores.std(0, ddof=1)
    keys = ["ROC-AUC", "F1", "Accuracy", "Precision", "Recall"]
    return {k: f"{m:.4f}±{s:.4f}" for k, m, s in zip(keys, means, stds)}

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

#Загрузка модели Qwen 3.0 4B Instruct
model_name = "Qwen/Qwen3-4B-Instruct-2507"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Загрузка токенизаторов
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Загрузка модель с 16-битной квантизацией и автоматическим распределением по устройствам
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    device_map=device,
    attn_implementation="sdpa"
)

if torch.cuda.is_available():
    print(f"Использовано памяти: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Using device: cuda


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

Использовано памяти: 8.04 GB


In [ ]:
def create_prompt(row, feature_names, target_name, dataset_name, prompt_config, tokenizer, include_answer=False):

    labels_str = "', '".join(prompt_config['labels'])

    system_prompt = (
        f"You are a classifier. Dataset: {dataset_name}. Task: {prompt_config['task']}.\n"
        f"Answer with only one word from: '{labels_str}'."
    )

    user_message = (
        f"Dataset: {dataset_name}. {prompt_config['entity']} information: "
        f"{row_to_text_template(row, feature_names)}\n"
        f"{prompt_config['question']}"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_message},
    ]

    # Если это данные для обучения, добавляем ответ ассистента
    if include_answer:
        target_value = row[target_name]
        answer = prompt_config['labels'][int(target_value)]
        messages.append({"role": "assistant", "content": answer})

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
def predict_single_with_prob(prompt, feature_names, target_name, dataset_name, prompt_config, model, tokenizer, device, max_new_tokens=5):
    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model.generate(
            model_inputs.input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            output_scores=True,
            return_dict_in_generate=True,
        )

    # Декодирование текста
    generated_ids  = outputs.sequences[0][model_inputs.input_ids.shape[1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip().lower()

    # Извлечение вероятностей для всех классов
    first_token_logits = outputs.scores[0][0]

    labels_ids = []
    for labels_name in prompt_config['labels']:
        labels_id = tokenizer.encode(labels_name, add_special_tokens=False)[0]
        labels_ids.append(labels_id)

    labels_logits = torch.stack([first_token_logits[cid] for cid in labels_ids])
    probs = F.softmax(labels_logits, dim=0)

    # Возвращаем ответ и вероятности всех классов
    return response, probs.cpu().numpy()

In [ ]:
# Тест
dataset_name = "Jungle"
test_row = test_df.iloc[0]
current_config = DATASET_CONFIGS[dataset_name]["prompt_config"]

test_prompt = create_prompt(
    test_row,
    feature_names,
    target_name,
    dataset_name,
    current_config,
    tokenizer
)

response, probs = predict_single_with_prob(
    prompt=test_prompt,
    feature_names=feature_names,
    target_name=target_name,
    dataset_name=dataset_name,
    prompt_config=current_config,
    model=base_model,
    tokenizer=tokenizer,
    device=device
)

print(f"Dataset: {dataset_name}")
print(f"Prompt Sample: {test_prompt[:120]}")
print(f"Model Response: '{response}'")

prob_dict = dict(zip(current_config['labels'], probs))

print("Probabilities:")
for label, p in prob_dict.items():
    print(f"  {label:15}: {p:.4f}")

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


Dataset: Jungle
Prompt Sample: <|im_start|>system
You are a classifier. Dataset: Jungle. Task: Predict the endgame result of Jungle Chess.
Answer with 
Model Response: 'draw'
Probabilities:
  white_win      : 0.0000
  draw           : 1.0000
  black_win      : 0.0000


In [ ]:
def evaluate_dataset(model, df, feature_names, target_name, dataset_name, prompt_config, task_type, tokenizer, device, desc="Eval"):
    y_true, y_pred, y_prob = [], [], []

    for _, row in tqdm(df.iterrows(), total=len(df), desc=desc, leave=False):
        prompt = create_prompt(row, feature_names, target_name, dataset_name, prompt_config, tokenizer)

        response, probs = predict_single_with_prob(
            prompt, feature_names, target_name, dataset_name,
            prompt_config, model, tokenizer, device
        )

        y_true.append(int(row[target_name]))
        y_pred.append(parse_prediction(response, prompt_config['labels']))

        if task_type == "binary":
            y_prob.append(float(probs[1]))
        else:
            y_prob.append(probs)


    return np.array(y_true), np.array(y_pred), np.array(y_prob)

# 3. Fine-tuning с LoRA

## Подготовка данных для Fine-tuning

In [ ]:
def balance_dataset(train_df, target_name, labels_info, seed=42):
    labels_counts = train_df[target_name].value_counts()

    max_count = labels_counts.max()
    balanced_dfs = []

    for cls in labels_counts.index:
        df_labels = train_df[train_df[target_name] == cls]
        if len(df_labels) < max_count:
            df_upsampled = resample(
                df_labels, replace=True, n_samples=max_count, random_state=seed
            )
            balanced_dfs.append(df_upsampled)
        else:
            balanced_dfs.append(df_labels)

    df_balanced = pd.concat(balanced_dfs)
    return df_balanced.sample(frac=1, random_state=seed).reset_index(drop=True)

In [ ]:
def create_training_dataset(all_splits, seed=42):
    all_texts = []
    for name, (train_df, _, _, feature_names, target_name) in all_splits.items():
        prompt_config = DATASET_CONFIGS[name]["prompt_config"]

        train_df_balanced = balance_dataset(train_df, target_name, prompt_config['labels'], seed=seed)

        for _, row in tqdm(train_df_balanced.iterrows(), total=len(train_df_balanced), desc=f"Датасет: {name}", leave=False):
            text = create_prompt(
                row, feature_names, target_name,
                dataset_name=name,
                prompt_config=prompt_config,
                tokenizer=tokenizer,
                include_answer=True
            )
            all_texts.append(text)

    import random
    random.seed(seed)
    random.shuffle(all_texts)
    print(f"Общее количество примеров в тренировочном пуле: {len(all_texts)}")
    return all_texts

all_texts = create_training_dataset(all_splits)
all_texts[:10]

Датасет: Bank:   0%|          | 0/47906 [00:00<?, ?it/s]

Датасет: Blood:   0%|          | 0/684 [00:00<?, ?it/s]

Датасет: California:   0%|          | 0/12380 [00:00<?, ?it/s]

Датасет: Car:   0%|          | 0/2904 [00:00<?, ?it/s]

Датасет: Credit_G:   0%|          | 0/840 [00:00<?, ?it/s]

...

Датасет: Heart:   0%|          | 0/608 [00:00<?, ?it/s]

Датасет: Income:   0%|          | 0/44584 [00:00<?, ?it/s]

Датасет: Jungle:   0%|          | 0/41511 [00:00<?, ?it/s]

Общее количество примеров в тренировочном пуле: 152017


["<|im_start|>system\nYou are a classifier. Dataset: Jungle. Task: Predict the endgame result of Jungle Chess.\nAnswer with only one word from: 'white_win', 'draw', 'black_win'.<|im_end|>\n<|im_start|>user\nDataset: Jungle. Game Position information: The value of white_piece0_strength is 7. The value of white_piece0_file is 1. The value of white_piece0_rank is 2. The value of black_piece0_strength is 0. The value of black_piece0_file is 1. The value of black_piece0_rank is 7.\nWhat is the game result?<|im_end|>\n<|im_start|>assistant\nblack_win<|im_end|>\n<|im_start|>assistant\n",
 "<|im_start|>system\nYou are a classifier. Dataset: Car. Task: Predict car evaluation.\nAnswer with only one word from: 'unacceptable', 'acceptable', 'good', 'very good'.<|im_end|>\n<|im_start|>user\nDataset: Car. Car information: The category of buying is high. The category of maint is high. The category of doors is 2. The category of persons is more. The category of lug_boot is small. The category of safet

In [ ]:
# Функция для токенизации текстовых последовательностей
def tokenize_fn(ex):
    return tokenizer(ex["text"], truncation=True, max_length=512, padding=False)

In [ ]:
# Основной процесс дообучения модели на нескольких задачах одновременно
def finetune_multitask_model(all_splits, num_epochs=3, batch_size=16):
    train_texts = create_training_dataset(all_splits)
    train_dataset = Dataset.from_dict({"text": train_texts})
    tokenized_dataset = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

    # Определение параметров адаптеров LoRA для эффективного использования памяти
    lora_config = LoraConfig(
        r=16, lora_alpha=32,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                        "gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model_lora = get_peft_model(base_model, lora_config)
    model_lora.print_trainable_parameters()

    # Настройка аргументов и стратегии процесса обучения
    training_args = TrainingArguments(
        output_dir=base_output_dir,
        num_train_epochs=num_epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        bf16=True,
        logging_steps=50,
        save_strategy="epoch",
        optim="adamw_torch_fused",
        warmup_steps=100,
        report_to="none",
        group_by_length=True
    )

    # Инициализация объекта для управления процессом обучения
    trainer = Trainer(
        model=model_lora,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    )

    # Запуск цикла обучения и фиксация времени выполнения
    t0 = time.time()
    trainer.train()
    print(f"Обучение успешно завершено за {(time.time()-t0)/60:.1f} минут.")

    # Сохранение параметров обученных адаптеров в указанную директорию
    model_lora.save_pretrained(base_output_dir)
    torch.cuda.empty_cache()

In [ ]:
def evaluate_checkpoints_on_val(all_splits):
    checkpoints = sorted(
        glob.glob(f"{base_output_dir}/checkpoint-*"),
        key=lambda x: int(re.search(r'checkpoint-(\d+)', x).group(1))
    )

    # Список для хранения результатов всех чекпоинтов
    all_results = []

    for cp in checkpoints:
        checkpoint_name = cp.split('/')[-1]
        steps = checkpoint_name.split('-')[-1]

        model_cp = PeftModel.from_pretrained(base_model, cp).to(device).eval()

        # Словари для накопления метрик по всем датасетам
        cp_metrics = {"ROC-AUC": [], "F1": [], "Accuracy": [], "Precision": [], "Recall": []}

        for name, (_, val_df, _, feature_names, target_name) in all_splits.items():
            config = DATASET_CONFIGS[name]

            # Получение предсказаний
            yt, yp, ypr = evaluate_dataset(
                model_cp, val_df, feature_names, target_name,
                name, config["prompt_config"], config["task_type"],
                tokenizer, device, desc=f"{checkpoint_name}: {name}"
            )

            # Расчет метрик
            m = compute_metrics(yt, yp, list(ypr), config["task_type"])
            for key in cp_metrics.keys():
                cp_metrics[key].append(m[key])

        # Вычисление среднего значения по всем задачам
        result_row = {
            "Checkpoint": checkpoint_name.split('-')[-1], # Номер шага
            "ROC-AUC": np.mean(cp_metrics["ROC-AUC"]),
            "F1": np.mean(cp_metrics["F1"]),
            "Accuracy": np.mean(cp_metrics["Accuracy"]),
            "Precision": np.mean(cp_metrics["Precision"]),
            "Recall": np.mean(cp_metrics["Recall"])
        }
        all_results.append(result_row)

        # Очистка памяти
        del model_cp; torch.cuda.empty_cache()

    df_results = pd.DataFrame(all_results)
    best_idx = df_results['ROC-AUC'].idxmax()
    best_cp_step = df_results.loc[best_idx, 'Checkpoint']

    return df_results, f"{base_output_dir}/checkpoint-{best_cp_step}"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

finetune_multitask_model(all_splits, num_epochs=3, batch_size=16)

df_results, best_model_path = evaluate_checkpoints_on_val(all_splits)

Mounted at /content/drive


Датасет: Bank:   0%|          | 0/47906 [00:00<?, ?it/s]

Датасет: Blood:   0%|          | 0/684 [00:00<?, ?it/s]

Датасет: California:   0%|          | 0/12380 [00:00<?, ?it/s]

Датасет: Car:   0%|          | 0/2904 [00:00<?, ?it/s]

...

checkpoint-14253: Credit_G:   0%|          | 0/200 [00:00<?, ?it/s]

checkpoint-14253: Diabetes:   0%|          | 0/154 [00:00<?, ?it/s]

checkpoint-14253: Heart:   0%|          | 0/184 [00:00<?, ?it/s]

checkpoint-14253: Income:   0%|          | 0/9769 [00:00<?, ?it/s]

checkpoint-14253: Jungle:   0%|          | 0/8964 [00:00<?, ?it/s]

In [ ]:
print(f"Лучший чекпоинт на основе ROC-AUC: {best_model_path}")

Лучший чекпоинт на основе ROC-AUC: /content/drive/MyDrive/finetuned_qwen_multitask/checkpoint-9502


In [ ]:
df_results

,Checkpoint,ROC-AUC,F1,Accuracy,Precision,Recall
0,4751,0.849590,0.640753,0.752573,0.641586,0.673460
1,9502,0.870662,0.705923,0.814114,0.693084,0.732935
2,14253,0.869987,0.687721,0.815484,0.701719,0.686845


Время обучения: ~ 4 ч 37 мин

Время выборки модели: 10 ч

Использование оперативная памяти графического процесса: 50.4/80GB

Графический процессор A100 80GB